# HydroSeason AOI Rainfall Fetch

This notebook shows how to fetch monthly rainfall for an area of interest and feed it directly into HydroSeason. SILO is useful for Australian AOIs; ERA5 is useful globally.

Supported AOI vector inputs include GeoJSON, SHP, KML, KMZ, GPKG, GPCK, and other formats readable by GeoPandas. Use a cache directory for repeated runs.

In [ ]:
from pathlib import Path

from hydroseason import (
    delineate_monthly_dataframe,
    generate_html_report,
    get_monthly_silo_rainfall,
    get_monthly_variable,
    load_vector,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

AOI = ROOT / "data" / "fitzroy_catchment.geojson"
AOI

## Load the AOI

`load_vector()` normalises vector loading for both fetchers. Substitute your own `.shp`, `.kml`, `.kmz`, `.gpkg`, or `.gpck` path here.

In [ ]:
gdf = load_vector(AOI)
gdf.to_crs("EPSG:4326").total_bounds

## SILO Monthly Rainfall (Australia)

SILO monthly rainfall uses public annual NetCDF files and returns AOI-averaged monthly totals in millimetres. The first run downloads annual NetCDF files; later runs reuse the Parquet result cache when `cache_dir` is set.

In [ ]:
RUN_SILO_FETCH = False

if RUN_SILO_FETCH:
    silo_monthly = get_monthly_silo_rainfall(
        gdf,
        start_year=1985,
        end_year=2023,
        cache_dir=ROOT / "data" / "silo_cache",
    )
    silo_monthly.to_csv(OUTPUT / "silo_monthly_rainfall.csv", index=False)
    silo_monthly.head()
else:
    print("Set RUN_SILO_FETCH = True to download SILO monthly rainfall.")

## ERA5 Rainfall (Global)

ERA5 fetch reads the public Google Cloud Zarr archive. Rainfall is summed from hourly metres to monthly millimetres.

In [ ]:
RUN_ERA5_FETCH = False
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

if RUN_ERA5_FETCH:
    era5_monthly = get_monthly_variable(
        path=ERA5_ZARR,
        gdf=gdf,
        start_year=1985,
        end_year=2023,
        variable="rainfall",
        cache_dir=ROOT / "data" / "era5_cache",
    )
    era5_monthly.to_csv(OUTPUT / "era5_monthly_rainfall.csv", index=False)
    era5_monthly.head()
else:
    print("Set RUN_ERA5_FETCH = True to download ERA5 rainfall.")

## Run HydroSeason After Fetch

Once you have `silo_monthly` or `era5_monthly`, the analysis step is identical to a local CSV workflow.

In [ ]:
# Example after a fetch has run:
# artifacts = delineate_monthly_dataframe(silo_monthly)
# result = artifacts.result
# report_path = generate_html_report(artifacts, OUTPUT / "aoi_fetch_report.html")
# result[["Date", "Rainfall_mm", "SeasonType", "Hydro_Year"]].head()

## CLI Equivalents

```bash
hydroseason fetch --source silo --vector data/fitzroy_catchment.geojson --start-year 1985 --end-year 2023 --cache-dir data/silo_cache --output output/silo_monthly_rainfall.csv

hydroseason fetch --source era5 --path gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3 --vector data/fitzroy_catchment.geojson --start-year 1985 --end-year 2023 --variable rainfall --cache-dir data/era5_cache --output output/era5_monthly_rainfall.csv
```

## Fetch-Enabled YAML

A config can fetch and run the pipeline in one command. When `fetch.enabled` is true, `input.csv_path` is optional.

```yaml
output:
  output_csv: output/silo_hydroseason_results.csv

fetch:
  enabled: true
  source: silo
  vector_path: data/fitzroy_catchment.geojson
  start_year: 1985
  end_year: 2023
  cache_dir: data/silo_cache
```